In [1]:
!pip install -qU transformers langchain-huggingface langchain-core langchain-community pydantic torch accelerate jupyter ipywidgets


# LangChain & Local Llama 3.1: Jira Ticket Assistant

This notebook acts as a complete application orchestrating an AI assistant using a local Hugging Face Model (`meta-llama/Llama-3.1-8B-Instruct`) connected natively through **LangChain**. 

**Core functionalities include:**
- **Dual-Identity Chain Routing**: Conditionally handles conversations normally OR switches to a strict Jira ticket formatting persona.
- **Long-Term Context via Summarization**: Compresses large historical message spans iteratively with overlapping deduplication to prevent hitting context limits.
- **Automated Self-Correction**: Implements agentic LLM retry loops to correct their own output if formatting schemas fail internal system validations.
- **Schema Extraction to JSON**: Seamlessly leverages Pydantic-powered parsers to export formal conversation outputs directly into application-ready JSON payloads.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# Define the model ID for the Qwen language model we want to use from Hugging Face.
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# You will need a Hugging Face token logged in via CLI for Llama 3.1
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
# Ensure pad_token exists (avoids generation warnings in many decoder-only models)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", # Automatically uses your GPU if available
    dtype="auto" # Optimizes memory usage
)

# --- Create the Transformers Pipeline ---
# Note: For local pipelines, the task is strictly "text-generation"
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    temperature=0.1,
    max_length=None,
    return_full_text=False # Prevents the model from repeating your prompt
)

# --- Wrap it for LangChain ---
# Convert the Hugging Face pipeline into a LangChain LLM
local_llm = HuggingFacePipeline(pipeline=pipe)

# CRITICAL: We STILL need ChatHuggingFace to format the history for Llama!
chat_model = ChatHuggingFace(llm=local_llm)

In [4]:
ROLE_STOP_MARKERS = ("Human:", "System:", "AI:")

def parse_output(text: str) -> str:
    """Trim role-tag artifacts from generated text and return clean output."""
    # Initialize the cleaned text with the raw string
    cleaned = text
    for stop_word in ROLE_STOP_MARKERS:
        marker_index = cleaned.find(stop_word)
        if marker_index != -1:
            cleaned = cleaned[:marker_index]
            break
    # Strip any trailing whitespace around the response
    return cleaned.strip()

## 1. Session Memory & Short-Term Buffer

This section establishes an in-memory store for tracking different user chat sessions. By utilizing a sliding window approach, maintain recent context (a short-term buffer) using LangChain's `ChatMessageHistory`. Every user session is uniquely tracked so that context is not lost across multiple invocations.

In [5]:
# In-memory storage for different chat sessions.
store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Return per-session chat history, creating it on first access."""
    # If the session is uninitialized, instantiate a new ChatMessageHistory
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    # Cache and retrieve the requested session's state directly
    return store[session_id]

## 2. Jira Assistant Chat Chain

Here we define a specialized LangChain architecture tailored for generating structured Jira tickets. A strict system prompt is used to force the language model into the persona of a Jira Ticket Formatter, guaranteeing that explicit headers (`Title:`, `Type:`, `Priority:`, etc.) are consistently generated without any conversational filler or hallucination.

### Utility Functions: Validating and Repairing Outputs

To ensure generated Jira tickets are flawless, we implement validation functions:
- **`validate_jira_ticket_format`**: Scans the model output iteratively for all required Jira sections to verify structural integrity.
- **`overwrite_last_ai_message`**: Mutates the LangChain memory to overwrite formatting mistakes with repaired text, preventing models from learning from broken previous outputs.
- **`validate_and_repair_jira`**: Coordinates validation checks and safely retries generation up to a strict limit, effectively self-healing formatting errors.

In [6]:
# --- Build the LCEL Chain ---
# We use MessagesPlaceholder to tell LangChain exactly where to inject the memory
# Define strict system prompt as a variable for cleaner code
jira_system_instructions = """You are a strict Jira Ticket Formatter. 

RULES:
- Your goal is to help the user write detailed, well-structured Jira tickets (Bugs, Tasks, Stories, or Epics).
- Must include exactly these headers: Title:, Type:, Priority:, Description:, Labels:, Attachments:, Reporter:, Acceptance Criteria:.
- Be concise, direct, highly structured, and professional.
- REVISIONS: If the user asks to change or update something, you MUST rewrite the ENTIRE ticket from the previous turn. Apply their changes, but strictly keep all other previously established information intact. Do not erase previous details.

CRITICAL RULES:
- DO NOT ask follow-up questions.
- DO NOT output any multiple-choice options.
- DO NOT output ANY introductory, prefatory, or concluding text. 
- NEVER say things like "Here is the ticket" or acknowledge the user's prompt. 
- You must output ONLY the ticket content. Your response MUST start exactly with "Title:".
"""

jira_prompt = ChatPromptTemplate.from_messages([
    ("system", jira_system_instructions),
    ("system", "Conversation summary (if any):\n{summary_context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

jira_chain = jira_prompt | chat_model | StrOutputParser() | parse_output

jira_chat_bot = RunnableWithMessageHistory(
    jira_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

## 3. Self-Correction & Format Repair Chain

Even with highly specific prompts, LLMs may sometimes omit requested fields. This section initializes a fallback secondary chain (`jira_repair_chain`). If a generated Jira draft fails the aforementioned checks, this repair pipeline is directly injected to fix missing fields while preserving user intent and original structure.

In [7]:
REPAIR_STATS = {
    "jira_repair": 0,
    "json_repair": 0,
}

REQUIRED_JIRA_HEADERS = [
    "Title:",
    "Type:",
    "Priority:",
    "Description:",
    "Labels:",
    "Attachments:",
    "Reporter:",
    "Acceptance Criteria:",
]

jira_repair_system_instructions = """
You are a strict Jira ticket repair assistant.
Fix the provided Jira ticket so it follows the required structure exactly.

Rules:
- Output only the ticket text.
- Start with Title:.
- Include all required headers exactly once.
- Preserve user intent and existing facts.
        - Do not add unrelated details.
"""

jira_repair_prompt = ChatPromptTemplate.from_messages([
    ("system", jira_repair_system_instructions),
    ("human", """Required headers:\n{required_headers}\n\nUser intent:\n{user_input}\n\nDraft ticket:\n{ticket_text}\n\nRepaired ticket: """),
])


jira_repair_chain = jira_repair_prompt | chat_model | StrOutputParser() | parse_output

def validate_jira_ticket_format(ticket_text: str) -> tuple[bool, list[str]]:
    """Validate Jira output structure and report missing required headers."""
    # Normalize lines and ignore blanks before structural checks.
    lines = [line.strip() for line in ticket_text.splitlines() if line.strip()]
    # If the text is completely empty or just whitespace, immediately return failure
    if not lines:
        return False, REQUIRED_JIRA_HEADERS.copy()

    # Identify which required headers are missing from the output lines
    missing_headers = [
        header for header in REQUIRED_JIRA_HEADERS
        if not any(line.startswith(header) for line in lines)
    ]

    # Verify that the ticket accurately starts with the Title section
    starts_ok = lines[0].startswith("Title:")
    # The ticket is valid if it starts correctly and has no missing headers
    is_valid = starts_ok and not missing_headers
    return is_valid, missing_headers

def overwrite_last_ai_message(session_id: str, new_text: str) -> None:
    """Replace the most recent AI message in history with repaired content."""
    # Fetch existing chat history corresponding to the active session id
    history = get_session_history(session_id)
    # Loop backward across the history tracing for the latest generated artifact
    for message in reversed(history.messages):
        # Match against AI generated nodes only
        if getattr(message, "type", "") == "ai":
            # Safely replace its contents recursively across memory
            message.content = new_text
            # Cease operation immediately post overwrite
            return

def validate_and_repair_jira(
    ticket_text: str,
    user_input: str,
    session_id: str,
    max_retries: int = 5,
    ) -> tuple[str, bool]:
    """Run format validation and apply limited self-repair passes if needed."""
    # Initialize candidate parsing block and track validation events sequentially 
    candidate = ticket_text
    repaired = False

    # Execute retry validation up to maximum threshold 
    for attempt in range(max_retries + 1):
        # Exit early once candidate satisfies required structure.
        is_valid, _ = validate_jira_ticket_format(candidate)
        # Verify component compliance successfully validating pipeline properties 
        if is_valid:
            return candidate, repaired

        # Cease retry evaluation if thresholds met without proper resolution 
        if attempt == max_retries:
            break

        # Render string layouts mapping formatting issues safely
        required_headers_text = "\n".join(REQUIRED_JIRA_HEADERS)
        # Re-invoke execution safely injecting context explicitly to resolve validation constraints 
        candidate = jira_repair_chain.invoke(
            {
                "required_headers": required_headers_text,
                "user_input": user_input,
                "ticket_text": candidate,
            }
        ).strip()
        repaired = True

    # Assert persistence modifications cleanly on pipeline validations correctly 
    if repaired:
        REPAIR_STATS["jira_repair"] += 1
        # Sync repaired artifacts directly into session layer successfully resolving memory
        overwrite_last_ai_message(session_id, candidate)

    return candidate, repaired

## 4. Normal Chat Chain

For non-Jira-related user inputs, a secondary relaxed conversational chain acts as a typical AI assistant to answer general questions directly without enforcing arbitrary constraints on the user.

In [8]:
normal_system_instructions = """You are a helpful and concise AI assistant.
- Answer clearly and directly.
- Ask follow-up questions only when needed to clarify user intent.
- Do not force Jira ticket formatting unless the user explicitly asks for Jira/template/ticket creation.
"""

normal_prompt = ChatPromptTemplate.from_messages([
    ("system", normal_system_instructions),
    ("system", "Conversation summary (if any):\n{summary_context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

normal_chain = normal_prompt | chat_model | StrOutputParser() | parse_output

normal_chat_bot = RunnableWithMessageHistory(
    normal_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

## 5. Routing Mechanism: The LLM Judge

When user intent is highly unstructured, heuristic strategies may fail. This section prepares an **LLM Judge Chain**—an auxiliary model pathway entirely devoted to assessing routing. It reliably assigns the user input into either the `jira` or `normal` execution paths. Although this approach contributes to latency via a secondary inference step, it substantially improves context reliability when managing complex intent.

In [9]:

judge_system_instruction = """
    You are a routing judge.
    Given the user message and a tentative route, return exactly one final label: jira or normal.
    Choose jira only if the user is clearly asking for Jira ticket/template creation or revision.
    Otherwise return normal.
    Do not output anything except jira or normal.
"""
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", judge_system_instruction ),
    ("human", "User message:\n{input}\n\nTentative route: {tentative_route}\n\nFinal route:",),
])

judge_chain = judge_prompt | chat_model | StrOutputParser()

## 6. Long-Term Memory Compression (Summary Module)

As conversational history extends, holding onto the entire transcription footprint degrades both token usage and output reasoning capability. This architecture overcomes context exhaustion dynamically by building an auto-triggering recursive summarization routine that compresses legacy conversation states into succinct semantic bullets.

### Utility Functions: Memory Summarization Mechanics

The memory loop uses numerous state-tracking heuristics:
- **Token & Event Counters (`history_token_count`, `user_turn_count`)**: Calculates memory thresholds to activate context compression lazily without manual intervention.
- **Vector Context Matching (`normalize_text`, `remove_summary_overlap`)**: Deduplicates redundant semantic overlaps logically bridging the short-term buffer with long-term summaries.
- **Slicers & Refreshers (`refresh_summary`)**: Isolates raw message scopes logically before delegating contextual reconstruction directly to the embedded pipeline.

In [10]:
SUMMARY_EVERY_TURNS = 5
SUMMARY_TOKEN_THRESHOLD = 1200
SHORT_TERM_BUFFER_MESSAGES = 12
SUMMARY_SOURCE_MAX_MESSAGES = 40

summary_store: dict[str, str] = {}

summary_memory_instruction = """
    You are a conversation memory assistant.
    Update and compress the memory summary for one chat session.
    Rules:
    - Keep only durable facts, user preferences, decisions, constraints, and unresolved items.
    - Remove small talk and duplicate details.
    - Do not include details already present in the recent short-term buffer.
    - Keep it concise (max 8 bullets).
    - For Jira mode, preserve ticket facts (title intent, type, priority, labels, reporter, acceptance criteria decisions).
    Return plain text bullet points only.
"""

summary_prompt = ChatPromptTemplate.from_messages([
    ("system",summary_memory_instruction),
    ("human","Route: {route}\n\nCurrent summary:\n{current_summary}\n\nLonger-term messages (excluding recent buffer):\n{summary_source_messages}\n\nUpdated summary:"),
])

summary_chain = summary_prompt | chat_model | StrOutputParser()

In [11]:
import re

def history_token_count(session_id: str) -> int:
    """Estimate token usage across the full message history of a session."""
    
    history = get_session_history(session_id)
    # Dynamically evaluate the overall token expense of all recorded transactions
    return sum(
        # Invoke HuggingFace tokenizer recursively across the unencoded historical artifacts
        len(tokenizer.encode(str(getattr(msg, "content", "") or ""), add_special_tokens=False))
        for msg in history.messages
    )

def user_turn_count(session_id: str) -> int:
    """Count user-originated turns in a session history."""

    history = get_session_history(session_id)
    # Filter constraints sequentially matching only prompts dictated natively as human requests 
    return sum(1 for msg in history.messages if getattr(msg, "type", "") == "human")

def normalize_text(text: str) -> str:
    """Normalize text for overlap checks using lowercase and punctuation stripping."""
    # Truncate out unneeded trailing whitespaces down to a uniformly flat dimension
    compact = re.sub(r"\s+", " ", text.strip().lower())
    # Purge symbolic punctuations entirely to stabilize literal deduplication checking
    return re.sub(r"[^a-z0-9 ]", "", compact)

def messages_to_text(messages: list) -> str:
    """Render messages into role-prefixed plain text lines."""
    
    lines: list[str] = []
    
    for msg in messages:
        # Resolve dynamic role pointers defaulting back safely against 'unknown' limits 
        role = getattr(msg, "type", "unknown")
        content = str(getattr(msg, "content", "")).strip()
        # Merge properties into logical lines
        lines.append(f"{role}: {content}")
        
    return "\n".join(lines)

def get_recent_messages_text(session_id: str, max_messages: int = SHORT_TERM_BUFFER_MESSAGES) -> str:
    """Serialize recent short-term window into role-prefixed plain text."""
    
    history = get_session_history(session_id)
    # Serialize historical boundaries backwards to cap text aggregation locally within limits
    return messages_to_text(history.messages[-max_messages:])

def get_summary_source_messages(
    session_id: str,
    skip_recent: int = SHORT_TERM_BUFFER_MESSAGES,
    max_messages: int = SUMMARY_SOURCE_MAX_MESSAGES,
    ) -> str:
    """Get older message text used as source when refreshing long-term summary."""
    
    history = get_session_history(session_id)
    
    older_messages = history.messages[:-skip_recent] if skip_recent > 0 else history.messages
    # Deny operation sequentially ensuring older iterations remain available
    if not older_messages:
        return ""
    # Process older buffers through logical serializers capping string evaluation globally 
    return messages_to_text(older_messages[-max_messages:])

def remove_summary_overlap(summary_text: str, session_id: str) -> str:
    """Remove summary bullets that duplicate recent short-term buffer content."""
    
    if not summary_text.strip():
        return ""

    recent_corpus = normalize_text(get_recent_messages_text(session_id))
    
    if not recent_corpus:
        return summary_text.strip()

    cleaned_lines: list[str] = []
    
    for raw_line in summary_text.splitlines():
        line = raw_line.strip()
    
        if not line:
            continue

        # Decouple symbols explicitly leaving pure content behind 
        bullet_body = line.lstrip("-*• ").strip()
        normalized = normalize_text(bullet_body)

        # Trigger logic ignoring overlap when limits are exceeded locally 
        if normalized and len(normalized) >= 20 and normalized in recent_corpus:
            continue

        # Append strings successfully parsing explicit symbols effectively protecting text formatting limits 
        cleaned_lines.append(line if line.startswith(("-", "*", "•")) else f"- {line}")

    return "\n".join(cleaned_lines).strip()

def refresh_summary(session_id: str, route: str) -> None:
    """Refresh summary by cadence or token threshold, then de-duplicate overlap."""
    turns = user_turn_count(session_id)
    token_count = history_token_count(session_id)

    if turns == 0:
        return

    should_refresh = (turns % SUMMARY_EVERY_TURNS == 0) or (token_count >= SUMMARY_TOKEN_THRESHOLD)
    if not should_refresh:
        return

    current_summary = summary_store.get(session_id, "(none)")
    summary_source_messages = get_summary_source_messages(session_id)
    if not summary_source_messages.strip():
        return

    try:
        updated_summary = summary_chain.invoke(
            {
                "route": route,
                "current_summary": current_summary,
                "summary_source_messages": summary_source_messages,
            }
        ).strip()
        cleaned_summary = remove_summary_overlap(updated_summary, session_id)
        if cleaned_summary:
            summary_store[session_id] = cleaned_summary
    except Exception:
        pass

def get_summary_context(session_id: str) -> str:
    """Return cleaned summary text to inject into prompts as long-term context."""
    summary = summary_store.get(session_id, "")
    return remove_summary_overlap(summary, session_id)

## 7. The Central Router

The Central Router orchestrates request triage between different execution spaces. Progressing from aggressive O(1) heuristic tracking mechanisms into deep programmatic LLM validation via the Judge Chain, it dynamically enforces confidence-based dispatch flows depending on ambiguity metrics.

### Utility Functions: Routing Analysis

To decouple analytical overhead from strict conversational duties, router execution operates against robust helper structures:
- **`track_metrics`**: A comprehensive telemetry proxy tracking component performance and pipeline latency.
- **`detect_route_confidence`**: A highly efficient pseudo-predictive classifier measuring user instructions iteratively over keyword signatures.
- **`detect_route_with_meta`**: The core router logic unifying heuristics, fallbacks, and the independent Judge engine safely regardless of prompt complexity.

In [12]:
from typing import Any
import time

ROUTE_JIRA = "jira"
ROUTE_NORMAL = "normal"

JIRA_ROUTE_KEYWORDS = (
    "jira", "ticket", "story", "epic", "bug", "task", "acceptance criteria",
    "labels", "priority", "reporter", "attachments", "description",
)
JIRA_ROUTE_VERBS = ("create", "write", "draft", "update", "revise", "format", "extract")
NORMAL_ROUTE_MARKERS = ("explain", "what is", "how does", "summarize", "compare", "hello", "hi")

router_system_instruction =  """
    You are a routing classifier.
    Return exactly one label: jira or normal.
    Choose jira only when the user's message is asking to create, edit, revise, structure, or extract a Jira ticket/template.
    Choose normal for all other messages.
    Do not output anything except jira or normal.
"""

router_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system_instruction),
    ("human", "{input}"),
])

router_chain = router_prompt | chat_model | StrOutputParser()

_The following functions were developed with the assistance of the Gemini 3.1 Pro large language model (Google, 2026)._

In [13]:
ROUTE_CONFIDENCE_THRESHOLD = 0.60

metrics_log: list[dict] = []

def track_metrics(event: str, payload: dict) -> None:
    """Store one telemetry event with timestamp for observability."""
    
    metrics_log.append({
       
        "timestamp": time.time(),   # Stamp telemetry event securely across runtime variables
        "event": event,             # Detail exactly which application event signaled correctly over runtime limits 
        "payload": payload,         # Aggregate any related objects recursively into tracked context maps 
    })

def count_hits(text: str, tokens: tuple[str, ...]) -> int:
    """Count how many token substrings are present in text."""
    return sum(1 for token in tokens if token in text)

def keyword_route(user_input: str) -> str:
    """Fallback route decision based on simple Jira keyword matching."""
    text = user_input.lower()
    # Forward routes efficiently verifying strictly upon raw footprint conditions
    return ROUTE_JIRA if count_hits(text, JIRA_ROUTE_KEYWORDS) > 0 else ROUTE_NORMAL

def detect_route_confidence(text: str) -> float:
    """Heuristic confidence score for Jira routing in range [0.05, 0.95]."""
    
    lowered = text.lower()

    # Log matching heuristics precisely counting string footprint signatures across dimensions
    keyword_hits = count_hits(lowered, JIRA_ROUTE_KEYWORDS)
    verb_hits = count_hits(lowered, JIRA_ROUTE_VERBS)
    normal_hits = count_hits(lowered, NORMAL_ROUTE_MARKERS)

    # Blend aggregated coefficients resolving to a scaled ratio bounded explicitly below 1.0  
    jira_signal = min(1.0, 0.18 * keyword_hits + 0.22 * verb_hits)
    normal_signal = min(1.0, 0.22 * normal_hits)
    confidence = 0.50 + (jira_signal - normal_signal) * 0.45
    
    return max(0.05, min(0.95, confidence))

def parse_route_label(raw: str) -> tuple[str, bool]:
    """Normalize model output to a route and flag ambiguous/invalid responses."""
    
    decision = (raw or "").strip().lower()
    has_jira = ROUTE_JIRA in decision
    has_normal = ROUTE_NORMAL in decision

    # Explicitly approve exact un-ambiguous labels seamlessly 
    if has_jira and not has_normal:
        return ROUTE_JIRA, False
    if has_normal and not has_jira:
        return ROUTE_NORMAL, False
    # Mark combined label sets as critically ambiguous requiring fallback 
    if has_jira and has_normal:
        return ROUTE_NORMAL, True

    # Reject unrecognized un-mappable outputs natively through standard defaults 
    return ROUTE_NORMAL, True

def detect_route_with_meta(user_input: str) -> tuple[str, float, bool, bool, str]:
    """Select route with confidence-aware judge/fallback and return metadata + short reason."""
    # Preset boolean toggles tracing component invocation safely
    judge_used = False
    fallback_used = False
    # Execute primary heuristic scanning initially
    confidence = detect_route_confidence(user_input)
    # Define primary reason boundaries to limit runtime edge cases logically
    reason = "router:uninitialized"

    try:
        router_raw = router_chain.invoke({"input": user_input})
        tentative_route, needs_judge = parse_route_label(router_raw)

        if needs_judge or confidence < ROUTE_CONFIDENCE_THRESHOLD:
            judge_used = True
            tentative_route = keyword_route(user_input)
            judge_raw = judge_chain.invoke({"input": user_input, "tentative_route": tentative_route})
            judged_route, still_ambiguous = parse_route_label(judge_raw)

            if still_ambiguous:
                fallback_used = True
                judged_route = keyword_route(user_input)
                reason = "route:fallback_after_ambiguous_judge"
            else:
                reason = "route:judge_selected_due_to_low_conf_or_ambiguity"
            route = judged_route
        else:
            route = tentative_route
            reason = "route:router_direct_high_confidence"

    except Exception as exc:
        fallback_used = True
        route = keyword_route(user_input)
        reason = f"route:exception_fallback:{type(exc).__name__}"

    track_metrics(
        "route_decision",
        {
            "route": route,
            "confidence": confidence,
            "judge_used": judge_used,
            "fallback_used": fallback_used,
            "reason": reason,
            "input_preview": user_input[:80],
        },
    )
    return route, confidence, judge_used, fallback_used, reason

## 8. Exporting Structured Pydantic Data

Drafting text-based tickets is highly intuitive, but real-world integrations demand programmatic abstractions. The final step constructs a powerful data pipeline parsing finalized draft representations down into rigorously validated JSON payloads immediately capable of hitting downstream services like the Jira API.

### Utility Functions: Structured Output Translation

- **`JiraTemplateSchema`**: An immutable Pydantic-powered constraint map acting as the structural blueprint for outgoing JSON definitions.
- **`validate_schema_payload`**: Defensively enforces and coerces datatype bounds across LLM extraction events instantly upon conversion.
- **`convert_json`**: Chains LangChain's native `JsonOutputParser` into the finalized ticket state logic. Should translation fail, an automatic fallback JSON-repair prompt triggers natively to resolve escaping flaws.

In [14]:
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# --- Define Your Target JSON Structure (The Parser) ---
class JiraTemplateSchema(BaseModel):
    title: str = Field(description="The title or summary of the Jira ticket")
    issue_type: str = Field(description="The type of the ticket (e.g., Bug, Task, Story, Epic)")
    priority: str = Field(description="The priority level (e.g., High, Medium, Low)")
    description: str = Field(description="The main description or body of the ticket")
    labels: List[str] = Field(description="A list of labels or tags associated with the ticket. If none, return an empty list.")
    attachments: str = Field(description="Details about any attachments, or 'None' if not applicable")
    reporter: str = Field(description="The name or placeholder of the person reporting the ticket")
    acceptance_criteria: str = Field(description="The criteria required for the ticket to be considered complete")

def validate_schema_payload(payload: dict) -> dict:
    """Validate extracted payload against schema and return normalized dict."""
    model_obj = JiraTemplateSchema.model_validate(payload)
    return model_obj.model_dump()

def _build_extraction_prompt(parser: JsonOutputParser) -> PromptTemplate:
    return PromptTemplate(
        template="""You are an expert data extraction algorithm.
        Extract the details from the provided Jira ticket text into JSON.
        Do not hallucinate or add extra information.

        {format_instructions}

        Ticket Text:
        {draft_text}
        """,
        input_variables=["draft_text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

def _build_repair_prompt(parser: JsonOutputParser) -> PromptTemplate:
    return PromptTemplate(
        template="""You repair JSON output for a Jira schema.
        Return one valid JSON object only. No markdown and no extra text.

        {format_instructions}

        Ticket Text:
        {draft_text}
        """,
        input_variables=["draft_text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

def convert_json(session_id: str = "jira_session_1") -> None:
    """Convert the latest Jira draft in session history to structured JSON."""
    conversion_start = time.perf_counter()
    user_history = get_session_history(session_id)

    if not user_history.messages:
        print("\nBot: No Jira draft found to convert.")
        track_metrics("json_conversion", {"session_id": session_id, "status": "no_draft"})
        return

    final_draft_text = user_history.messages[-1].content
    print("\nBot: Extracting your ticket into JSON...")

    parser = JsonOutputParser(pydantic_object=JiraTemplateSchema)
    extraction_chain = _build_extraction_prompt(parser) | chat_model | parser

    print("Extracting...\n")
    used_json_repair = False

    try:
        final_json_data = extraction_chain.invoke({"draft_text": final_draft_text})
    except Exception:
        print("Primary JSON parse failed. Running one repair pass...")
        REPAIR_STATS["json_repair"] += 1
        used_json_repair = True

        repair_chain = _build_repair_prompt(parser) | chat_model | StrOutputParser()
        repaired_raw = repair_chain.invoke({"draft_text": final_draft_text}).strip()

        try:
            repaired_payload = json.loads(repaired_raw)
            final_json_data = validate_schema_payload(repaired_payload)
        except Exception:
            print("JSON repair failed. Please continue refining the Jira draft and try again.")
            track_metrics(
                "json_conversion",
                {
                    "session_id": session_id,
                    "status": "failed",
                    "used_json_repair": used_json_repair,
                    "latency_ms": round((time.perf_counter() - conversion_start) * 1000, 2),
                },
            )
            return

    track_metrics(
        "json_conversion",
        {
            "session_id": session_id,
            "status": "success",
            "used_json_repair": used_json_repair,
            "latency_ms": round((time.perf_counter() - conversion_start) * 1000, 2),
        },
    )

    print("--- FINAL PARSED JSON ---")
    print(json.dumps(final_json_data, indent=2))

## 9. Context Garbage Collection & Telemetry Tests

Before application workflows yield processing back to users, memory integrity must be optimized. In this section, history limits are forcefully checked, trimming obsolete metadata out of context bounds. The section executes built-in diagnostic and system-wide routing smoke tests.

### Utility Functions: Garbage Collection

- **`trim_history`**: Implements dynamic garbage-collection routines. It rigorously removes older dialogue components beyond the static `keep_last_n` limit, guaranteeing deterministic LLM attention consistency preventing Out-of-Memory application crashes.

In [15]:
def trim_history(session_id: str, keep_last_n: int = 40, clear_summary_if_trimmed: bool = False) -> int:
    """Trim session history to the latest N messages and return removed count."""
    if keep_last_n < 0:
        raise ValueError("keep_last_n must be >= 0")

    history = get_session_history(session_id)
    total_messages = len(history.messages)
    if total_messages <= keep_last_n:
        return 0

    if keep_last_n == 0:
        del history.messages[:]
    else:
        del history.messages[:-keep_last_n]

    removed = total_messages - len(history.messages)
    summary_cleared = False
    if clear_summary_if_trimmed and session_id in summary_store:
        del summary_store[session_id]
        summary_cleared = True

    track_metrics(
        "history_trim",
        {
            "session_id": session_id,
            "removed": removed,
            "remaining": len(history.messages),
            "keep_last_n": keep_last_n,
            "summary_cleared": summary_cleared,
        },
    )
    return removed

## 10. CLI Interface: Central Control Loop

The final mechanism harmonizes all discrete components into a cohesive loop providing interactive capabilities to developers seamlessly across CLI terminal buffers.

### Core Interaction Loop
1. **Event Capture**: Intercepts physical inputs continuously while polling explicit termination and approval intents.
2. **Intent Classification**: Defers raw events via `detect_route_with_meta` capturing full programmatic metadata including confident ratios or LLM judge utilization.
3. **Orchestrated Output**: Selectively routes workloads seamlessly across `generate_jira_response` or `generate_normal_response` preserving logical separation of duties.
4. **Structured Delivery**: Intercepts finalizations (`looks good`, `approve`) programmatically pushing payloads through strict JSON validations.

In [16]:
jira_config: Any = {"configurable": {"session_id": "jira_session_1"}}
normal_config: Any = {"configurable": {"session_id": "normal_session_1"}}

EXIT_COMMANDS = {"exit"}
APPROVAL_COMMANDS = {"looks good", "im happy", "done", "approve"}
JIRA_SESSION_ID = "jira_session_1"
NORMAL_SESSION_ID = "normal_session_1"
HISTORY_KEEP_LAST_N = 40

# Reasoning summary controls (brief rationale only, not chain-of-thought).
SHOW_REASONING_TO_USER = True
MAX_REASONING_DISPLAY_CHARS = 160

def format_reasoning_summary(
    route: str,
    route_confidence: float,
    route_reason: str,
    judge_used: bool,
    fallback_used: bool,
    ) -> str:
    """Build a short, user-friendly why/confidence summary."""
    mode_label = "Jira ticket mode" if route == ROUTE_JIRA else "Normal chat mode"
    decision_bits = []
    if judge_used:
        decision_bits.append("judge used")
    if fallback_used:
        decision_bits.append("keyword fallback")
    decision_suffix = f" ({', '.join(decision_bits)})" if decision_bits else ""

    reason_preview = (route_reason or "n/a")[:MAX_REASONING_DISPLAY_CHARS]
    return (
        f"Why: Selected {mode_label}{decision_suffix}. "
        f"Reason: {reason_preview}. "
        f"Confidence: {route_confidence:.2f}"
    )

def generate_jira_response(user_input: str) -> tuple[str, str, float, int]:
    """Handle one Jira-routed turn and return response, session id, latency, trim count."""
    session_id = JIRA_SESSION_ID
    summary_context = get_summary_context(session_id)

    start_ms = time.perf_counter()
    response = jira_chat_bot.invoke(
        {"input": user_input, "summary_context": summary_context},
        config=jira_config,
    )
    latency_ms = round((time.perf_counter() - start_ms) * 1000, 2)

    response, was_repaired = validate_and_repair_jira(
        ticket_text=response,
        user_input=user_input,
        session_id=session_id,
        max_retries=1,
    )
    if was_repaired:
        print("Bot: Applied one format self-repair pass for Jira output.")
        track_metrics("jira_repair", {"session_id": session_id, "triggered": True})

    refresh_summary(session_id, route=ROUTE_JIRA)
    trimmed = trim_history(session_id, keep_last_n=HISTORY_KEEP_LAST_N)
    return response, session_id, latency_ms, trimmed

def generate_normal_response(user_input: str) -> tuple[str, str, float, int]:
    """Handle one normal-routed turn and return response, session id, latency, trim count."""
    session_id = NORMAL_SESSION_ID
    summary_context = get_summary_context(session_id)

    start_ms = time.perf_counter()
    response = normal_chat_bot.invoke(
        {"input": user_input, "summary_context": summary_context},
        config=normal_config,
    )
    latency_ms = round((time.perf_counter() - start_ms) * 1000, 2)

    refresh_summary(session_id, route=ROUTE_NORMAL)
    trimmed = trim_history(session_id, keep_last_n=HISTORY_KEEP_LAST_N)
    return response, session_id, latency_ms, trimmed

def handle_approval() -> bool:
    """Run approval-to-JSON flow. Returns True when chat loop should stop."""
    jira_history = get_session_history(JIRA_SESSION_ID)
    if not jira_history.messages:
        print("\nBot: I don't have a Jira draft yet. Continue chatting or ask for a Jira ticket.")
        return False

    print("\nBot: Great! Moving Jira draft to JSON parsing phase...\n")
    convert_json(JIRA_SESSION_ID)
    return True


In [17]:
print("Bot: Hi! I can chat normally or help build Jira tickets. Tell me what you need.")

while True:
    user_input = input("You: ").strip()

    print(f"\nInput: {user_input}\n")

    if not user_input:
        continue

    normalized_input = user_input.lower()
    if normalized_input in EXIT_COMMANDS:
        break

    if normalized_input in APPROVAL_COMMANDS:
        if handle_approval():
            break
        continue

    route, route_confidence, judge_used, fallback_used, route_reason = detect_route_with_meta(user_input)

    if SHOW_REASONING_TO_USER:
        reasoning_summary = format_reasoning_summary(
            route=route,
            route_confidence=route_confidence,
            route_reason=route_reason,
            judge_used=judge_used,
            fallback_used=fallback_used,
        )
        print(f"(reasoning summary): {reasoning_summary}")

    if route == ROUTE_JIRA:
        response, session_id, latency_ms, trimmed = generate_jira_response(user_input)
    else:
        response, session_id, latency_ms, trimmed = generate_normal_response(user_input)


    print(f"\nBot:\n{response.strip()}\n")

print(f"Repair stats: {REPAIR_STATS}")

Bot: Hi! I can chat normally or help build Jira tickets. Tell me what you need.

Input: What should I eat toady in Singapore? Give me 5 suggestion.

(reasoning summary): Why: Selected Normal chat mode (judge used). Reason: route:judge_selected_due_to_low_conf_or_ambiguity. Confidence: 0.50

Bot:
Singapore is known for its diverse food scene. Here are 5 popular food suggestions:

1. Chili Crab: A seafood dish made with mud crabs cooked in a sweet and spicy tomato-based sauce.
2. Hainanese Chicken Rice: A classic Singaporean dish featuring poached chicken served with fragrant rice cooked in chicken stock and chili sauce.
3. Char Kway Teow: Stir-fried noodles made with flat rice noodles, vegetables, and often meat or seafood.
4. Laksa: A spicy noodle soup made with a rich and flavorful broth, rice noodles, and various toppings.
5. Kaya Toast: Toast with butter and kaya, a sweet coconut jam made from coconut milk, sugar, and eggs.

Which one of these dishes sounds appealing to you?


Input